In [12]:
import matplotlib.pyplot as plt
import numpy as np
import mitsuba as mi
import pyvista as pv
import sionna
import tensorflow as tf
from sionna.rt import load_scene, Transmitter, Receiver, PlanarArray, PathSolver, Camera
from scipy.optimize import minimize
import h5py
from pathlib import Path
import sys

In [ ]:
# ============================================================================
# SYSTEM PARAMETERS AND CLASS DEFINITIONS
# ============================================================================

# --- SYSTEM PARAMETERS ---
CARRIER_FREQUENCY = 2.4e9  
BANDWIDTH = 20e6           
NUM_SUBCARRIERS = 64       
SUBCARRIER_SPACING = BANDWIDTH / NUM_SUBCARRIERS
WAVELENGTH = 3e8 / CARRIER_FREQUENCY
NUM_ANTENNAS_RX = 4

# --- Define subcarrier frequencies ---
subcarrier_indices = np.arange(NUM_SUBCARRIERS)
subcarriers = CARRIER_FREQUENCY + (subcarrier_indices - NUM_SUBCARRIERS/2) * SUBCARRIER_SPACING


class CSIAngleEstimator:
    """Angle of Arrival estimator using MUSIC algorithm"""
    def __init__(self, num_antennas, wavelength):
        self.num_antennas = num_antennas
        self.wavelength = wavelength
        self.antenna_spacing = wavelength / 2
        self.k = 2 * np.pi / wavelength
    
    def steering_vector(self, angle):
        positions = np.arange(self.num_antennas) * self.antenna_spacing
        return np.exp(1j * self.k * positions * np.sin(angle))
    
    def estimate_aoa_music(self, csi_matrix, num_sources=1):
        # csi_matrix must be [Antennas, Subcarriers]
        avg_csi = np.mean(csi_matrix, axis=1) # Average across subcarriers [Antennas]
        R = np.outer(avg_csi, avg_csi.conj())
        eigenvalues, eigenvectors = np.linalg.eigh(R)
        idx = eigenvalues.argsort()[::-1]
        noise_subspace = eigenvectors[:, idx[num_sources:]]
        angles = np.linspace(-np.pi/2, np.pi/2, 180)
        spectrum = np.zeros(len(angles))
        
        for i, angle in enumerate(angles):
            a = self.steering_vector(angle)
            spectrum[i] = 1.0 / (np.abs(
                a.conj() @ noise_subspace @ noise_subspace.conj().T @ a
            ) + 1e-10)
        
        peak_idx = np.argmax(spectrum)
        return angles[peak_idx], (angles, spectrum)

class FTMRangeEstimator:
    """Range estimator from CSI phase slope (FTM mechanism)"""
    def __init__(self, speed_of_light=3e8):
        self.c = speed_of_light
    
    def estimate_range_from_phase(self, csi_subcarriers, subcarrier_spacing):
        phase = np.unwrap(np.angle(csi_subcarriers))
        subcarrier_idx = np.arange(len(phase))
        coeffs = np.polyfit(subcarrier_idx * subcarrier_spacing, phase, 1)
        phase_slope = coeffs[0]
        toa = -phase_slope / (2 * np.pi)
        return abs(self.c * toa / 2)

class BilaterationSolver:
    """Hybrid localization using range and angle (Weighted Least Squares)"""
    @staticmethod
    def hybrid_localization(anchor_positions, ranges, angles, distance_std=0.5, angle_std=0.1):
        num_anchors = len(anchor_positions)
        distance_weight = 1 / (distance_std**2)
        angle_weight = 1 / (angle_std**2)
        
        def objective(pos):
            # pos is a 1D array of length 3: [x, y, z]
            # We need to compute distance from this point to each anchor
            
            # 1. Distance Errors (Ranging)
            # Broadcast: anchor_positions is (num_anchors, 3), pos is (3,)
            # We need to subtract pos from each row of anchor_positions
            estimated_dists = np.array([np.linalg.norm(pos - anchor_positions[i]) 
                                       for i in range(num_anchors)])
            
            range_errors_sq = (ranges - estimated_dists)**2
            error_d = distance_weight * np.sum(range_errors_sq)
            
            # 2. Angle Errors (AoA)
            error_a = 0.0 
            
            for i in range(num_anchors):
                if not np.isnan(angles[i]):
                    # Use only x,y coordinates for 2D angle calculation
                    direction = pos[:2] - anchor_positions[i, :2]
                    estimated_angle = np.arctan2(direction[1], direction[0])
                    angle_diff = angles[i] - estimated_angle
                    # Normalize angle difference to [-pi, pi]
                    angle_diff = np.arctan2(np.sin(angle_diff), np.cos(angle_diff))
                    error_a += angle_weight * (angle_diff**2)
            
            # Total Error
            total_error = error_d + error_a
            return float(total_error)
        
        # Initial guess: mean of anchor positions
        x0 = np.mean(anchor_positions, axis=0)
        
        result = minimize(objective, x0, method='BFGS') 
        return result.x

# --- IMPAIRMENT AND COMPENSATION FUNCTIONS ---

def add_sfo(csi_matrix, subcarrier_indices, sfo_ppm=5):
    """Adds Sampling Frequency Offset (SFO) to the CSI matrix."""
    clock_mismatch = sfo_ppm * 1e-6
    phase_slope = 2 * np.pi * clock_mismatch
    sfo_phase = phase_slope * subcarrier_indices
    sfo_matrix = np.tile(np.exp(1j * sfo_phase), (csi_matrix.shape[0], 1))
    return csi_matrix * sfo_matrix

def add_phase_noise(csi_matrix, pn_std=0.5):
    """Adds Phase Noise (PN) / Common Phase Error (CPE)."""
    pn_shift = np.random.normal(0, pn_std) 
    pn_factor = np.exp(1j * pn_shift)
    return csi_matrix * pn_factor

def compensate_phase_noise(csi_matrix):
    """Compensates for CPE by removing the mean phase rotation."""
    total_phase = np.angle(csi_matrix)
    cpe = np.mean(total_phase)
    cpe_factor = np.exp(-1j * cpe)
    return csi_matrix * cpe_factor

print("✓ All necessary classes and functions defined.")

✓ All necessary classes and functions defined.


In [10]:
# ============================================================================
# SCENE SETUP AND RAY TRACING
# ============================================================================

no_preview = False

mi.set_variant("llvm_ad_mono_polarized")

scene_path = "../scene/scene_01.xml"

print("Loading scene with Sionna...")
scene = load_scene(scene_path)
print("Sionna scene loaded.")

lambda_half = 0.0625  # half wavelength spacing at 2.4 GHz

# AP as RX (home router, omnidirectional)
scene.rx_array = PlanarArray(
    num_rows=2,
    num_cols=2,
    vertical_spacing=lambda_half,
    horizontal_spacing=lambda_half,
    pattern="dipole",
    polarization="V"
)

# STA as TX (mobile device)
scene.tx_array = PlanarArray(
    num_rows=2,
    num_cols=2,
    vertical_spacing=lambda_half,
    horizontal_spacing=lambda_half,
    pattern="dipole",
    polarization="cross"
)

# --- Transmitter & Receiver Positions ---
tx_positions = [
    np.array([-2.5, 0.5, 0]),
    np.array([3.15, 0.6, -1.1]),
]
rx_positions = [
    np.array([-1.3, 0.65, -2.8]),
]

# --- Add TX/RX to Scene ---
tx_list, rx_list = [], []

for i, pos in enumerate(tx_positions):
    tx = Transmitter(name=f"tx_{i}", position=pos, display_radius=0.08)
    scene.add(tx)
    tx_list.append(tx)

for i, pos in enumerate(rx_positions):
    rx = Receiver(name=f"rx_{i}", position=pos, display_radius=0.08)
    scene.add(rx)
    rx_list.append(rx)

# Aim each transmitter toward each receiver
for tx in tx_list:
    for rx in rx_list:
        tx.look_at(rx)

print(f"Placed {len(tx_list)} transmitters and {len(rx_list)} receivers.\n")

camera_pos = [1.5, 1.0, 1.6]
camera_look = [1.5, 1.5, 1.3]

my_cam = Camera(
    position=camera_pos,
    look_at=camera_look,
)

solver_paths = PathSolver()
paths = solver_paths(scene, max_depth=5, los=True, specular_reflection=True, refraction=True)

num_paths = paths.tau.shape[-1]
print(f"Computed {num_paths} propagation paths per TX-RX link.\n")

Loading scene with Sionna...
Sionna scene loaded.
Placed 2 transmitters and 1 receivers.

Computed 111 propagation paths per TX-RX link.



In [11]:
# ============================================================================
# CSI CONVERSION, IMPAIRMENT FLOW, AND DERIVATION
# ============================================================================

# --- Data Extraction from Sionna ---
a, tau = paths.cir(normalize_delays=True, out_type="numpy")
a_snap = a[..., 0] 
tau_snap = tau

# Dynamically determine dimensions from paths object
num_rx, num_rx_ant, num_tx, num_tx_ant, num_paths = a_snap.shape

# Define Estimator/Solver instances
aoa_estimator = CSIAngleEstimator(num_rx_ant, WAVELENGTH)
range_estimator = FTMRangeEstimator()
solver = BilaterationSolver()

# Storage for derived localization inputs
all_estimated_ranges = []
all_estimated_angles = []
all_anchor_positions = []
true_target_position = rx_list[0].position.numpy()

print("\n" + "=" * 70)
print("Processing Each TX-RX Link (Generate -> Impair -> Compensate -> Derive)")
print("=" * 70)

for rx_idx in range(num_rx):
    rx_pos = rx_list[rx_idx].position.numpy()
    
    for tx_idx in range(num_tx):
        tx_pos = tx_list[tx_idx].position.numpy()
        
        # Calculate true AoA/Range (Ground Truth)
        true_range = np.linalg.norm(rx_pos - tx_pos)
        direction = rx_pos[:2] - tx_pos[:2]
        true_angle = np.arctan2(direction[1], direction[0])
        
        # 1. GENERATE IDEAL CSI
        H_freq_base = np.zeros((NUM_SUBCARRIERS, num_rx_ant), dtype=np.complex128)
        
        for rx_ant in range(num_rx_ant):
            for tx_ant in range(num_tx_ant):
                delays = tau_snap[rx_idx, tx_idx, :]
                powers = np.abs(a_snap[rx_idx, rx_ant, tx_idx, tx_ant, :])
                for l in range(num_paths):
                    H_freq_base[:, rx_ant] += powers[l] * np.exp(-1j * 2*np.pi* delays[l] * subcarriers)
        
        csi_mimo = H_freq_base.T 

        # 2. ADD NON-IDEALITIES (IMPAIRMENT)
        csi_sfo = add_sfo(csi_mimo, subcarrier_indices, sfo_ppm=5)
        csi_pn = add_phase_noise(csi_sfo, pn_std=0.5)

        # Add AWGN (SNR = 20 dB)
        snr_db = 20
        signal_power = np.mean(np.abs(csi_pn)**2)
        noise_power = signal_power * 10**(-snr_db/10)
        noise = np.sqrt(noise_power/2) * (np.random.randn(*csi_pn.shape) + 1j * np.random.randn(*csi_pn.shape))
        final_csi = csi_pn + noise
        
        # 3. APPLY COMPENSATION
        avg_csi_final = np.mean(final_csi, axis=0) 
        csi_compensated_for_aoa = compensate_phase_noise(final_csi)
        
        # 4. DERIVE LOCALIZATION PARAMETERS
        estimated_angle, _ = aoa_estimator.estimate_aoa_music(csi_compensated_for_aoa)
        estimated_range = range_estimator.estimate_range_from_phase(avg_csi_final, SUBCARRIER_SPACING)

        # Store results
        all_estimated_ranges.append(estimated_range)
        all_estimated_angles.append(float(estimated_angle))
        all_anchor_positions.append(tx_pos) 
        
        true_angle_scalar = float(true_angle)
        
        print(f"TX-{tx_idx} -> RX-{rx_idx}: Range Est={estimated_range:.2f}m (True={true_range:.2f}m)")
        print(f"              Angle Est={np.rad2deg(estimated_angle):.1f}° (True={np.rad2deg(true_angle_scalar):.1f}°)")


Processing Each TX-RX Link (Generate -> Impair -> Compensate -> Derive)
TX-0 -> RX-0: Range Est=0.21m (True=3.05m)
              Angle Est=0.5° (True=7.1°)
TX-1 -> RX-0: Range Est=0.59m (True=4.76m)
              Angle Est=-0.5° (True=179.4°)


C:\Users\pokem\AppData\Local\Temp\ipykernel_1504\1708984063.py:75: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  true_angle_scalar = float(true_angle)


In [ ]:
# ============================================================================
# FINAL LOCALIZATION DERIVATION AND RESULTS
# ============================================================================ 

print("\n" + "=" * 70)
print("Performing Hybrid Bilateration Localization")
print("=" * 70)

# Convert results to NumPy arrays
anchor_pos_array_final = np.array(all_anchor_positions)
ranges_final = np.array(all_estimated_ranges)
angles_final = np.array(all_estimated_angles)

# Perform hybrid bilateration
estimated_position = solver.hybrid_localization(
    anchor_pos_array_final,
    ranges_final,
    angles_final,
    distance_std=0.5, 
    angle_std=np.deg2rad(5)
)

# Calculate error
localization_error = np.linalg.norm(estimated_position - true_target_position)

print(f"\n{'='*60}")
print("FINAL LOCALIZATION RESULTS")
print(f"{'='*60}")
print(f"True Position:      [{true_target_position[0]:.2f}, {true_target_position[1]:.2f}, {true_target_position[2]:.2f}]")
print(f"Estimated Position: [{estimated_position[0]:.2f}, {estimated_position[1]:.2f}, {estimated_position[2]:.2f}]")
print(f"Localization Error: {localization_error:.3f} meters")
print(f"{'='*60}")


Performing Hybrid Bilateration Localization


ValueError: operands could not be broadcast together with shapes (2,) (2,3) 